In [1]:
# 必要なライブラリのimport
import pandas as pd
import statsmodels.formula.api as smf
import seaborn as sns

sns.set_theme()

/Users/MisayoMacBookPro/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
# irisのデータを読み込む
data = sns.load_dataset('iris')

data.tail()

,sepal_length,sepal_width,petal_length,petal_width,species
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica
149,5.9,3.0,5.1,1.8,virginica


まずは、説明変数を下記のようにそれぞれ、異なるように選択した3つの線形モデルを作成します。

- model1: がく片の幅
- model2: がく片の幅、花弁の長さ
- model3: がく片の幅、花弁の長さ、花弁の幅

statsmodels の ols 関数を使用して、下記のように線形モデルを作成できます。

<最小二乗法の実施>
smf.olsは、1つ目の引数に '目的変数 ~ 説明変数' という書き方をすることで、任意のモデルを作成できます。複数の説明変数を指定する場合は '目的変数 ~ 説明変数1+説明変数2+説明変数3' という書き方をします。

そして、fitによって、データから残差平方和が最小となる係数を求めています。

In [5]:
# 最小二乗法の実施（説明変数：sepal_widthのみ）
model1 = smf.ols(formula='sepal_length ~ sepal_width', data=data).fit()

In [6]:
# 最小二乗法の実施（説明変数：sepal_width・petal_length）
model2 = smf.ols(formula='sepal_length ~ sepal_width+petal_length', data=data).fit()

In [7]:
# 最小二乗法の実施（説明変数：sepal_width・petal_length・petal_width）
model3 = smf.ols(formula='sepal_length ~ sepal_width+petal_length+petal_width', data=data).fit()

<AICを計算する>
モデルごとにAICを計算します。

In [8]:
# AICを計算する
print('Model-1:',model1.aic)
print('Model-2:',model2.aic)
print('Model-3:',model3.aic)

Model-1: 369.9916713254629
Model-2: 99.02549981959152
Model-3: 82.64272058698862


AICのもっとも小さいモデルを選択しますので、model3が最適であるとわかりました。説明変数が3つある場合、以下のような式で線形モデルを表現できます。

sepal_length = (パラメータ0) + (パラメータ1) * (sepal_width) + (パラメータ2) * (petal_length) + (パラメータ3) * (petal_width)

最小二乗法（OLS）によって、パラメータ0から3を推定することになります。そのための何らかの処理は必要ありません。すでに smf.ols(...).fit() で推定が完了していますので、あとは、これで作成した model3 の詳細を確認すれば、4つのパラメータがわかります。

それでは、model3の詳細を確認してみましょう。

In [9]:
# AICが最小のmodel3の詳細を表示
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:           sepal_length   R-squared:                       0.859
Model:                            OLS   Adj. R-squared:                  0.856
Method:                 Least Squares   F-statistic:                     295.5
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           8.59e-62
Time:                        13:14:05   Log-Likelihood:                -37.321
No. Observations:                 150   AIC:                             82.64
Df Residuals:                     146   BIC:                             94.69
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        1.8560      0.251      7.401   

各種結果が記載されておりますが、表の上部分の重要な項目を確認しておきましょう。

Dep. Variable： 	   応答変数の名称
Model および Method：   パラメータの推定方法
No. Observations：     サンプルサイズ
F-statistic：           分散分析の結果（F比）
Prob (F-statistic)：    上記のF比から算出されたp値
Log-likelihood：        最大対数尤度
AIC：                   赤池情報量基準
BIC：                   ベイズ情報量基準

sepal_length = (パラメータ0) + (パラメータ1) * (sepal_width) + (パラメータ2) * (petal_length) + (パラメータ3) * (petal_width)とした場合のパラメータは表部分のcoef（=係数）列で確認できます。

表の他の部分の説明は以下の通り。

std err：           係数の標準誤差
t：                 係数のt値
P>|t|：             「係数が0である」を帰無仮説とした仮説検定のp値
[0.025 0.975] ：    係数の95％信頼区間

coefについて補足します。

coefこのモデルでは各係数coefについて、以下の仮説検定を行っています。

帰無仮説（H0）： パラメータ係数 = 0
対立仮説（H1）： パラメータ係数 <> 0

もし係数=0（対立仮説が棄却）であるならば、その項は式から消えてしまうため、「その変数はがく片の長さに影響を与えているとは言い切れない」ということになります。

この仮説検定については P>|t| 列にp値が記載されていますので、その値が事前に設定した有意水準以下であれば、「その変数はがく片の長さに影響を与えている」と言うことができます。これまで通り有意水準5％を採用した場合、この表からは、すべての説明変数が有意な影響を与えていることを読み取れます。

そして、各変数がプラスの影響を与えている（その値が大きいほどがく片は長くなる）のか、マイナスの影響を与えている（その値が大きいほどがく片は短くなる）のか、またその強さはどれほどか…。というのを知るために、説明変数の名前の行の coef 列を参照します。Intercept は切片（パラメータ0）です。